# 14. Confidence-weighted evaluation and confusion matrices

Two things Dr Hunter asked for on 29 September 2026.

**1. Confusion matrices** for every model, on both the within-dataset test set and the
cross-dataset (HASOC 2022) test set. He asked three times, because the aggregate
macro-F1 hides *which* class is failing and whether the failure is systematic.

**2. A confidence-weighted score** of his own definition. From his handwritten notes:

> For each test example, let the confidence difference be `D = P(Higher) - P(Lower)`,
> where `P(Higher)` and `P(Lower)` are the larger and smaller of the two class
> probabilities. Start a total at zero. If the classification is **correct**, add `+D`.
> If it is **incorrect**, subtract `D`. After all `N` examples, report the mean,
> `(1/N) * sum of (+/-)D_i`.

Reading the score:

| Value | Meaning |
|---|---|
| Close to **+1** | Many correct answers made with high confidence. Good. |
| Close to **0** | Either roughly equal numbers right and wrong, or many answers made with low confidence. |
| Close to **-1** | Many wrong answers made with **high** confidence. Very bad. |

The point of the metric is that it separates two failure modes that macro-F1 treats
identically: a model that is wrong but uncertain, and a model that is wrong and sure
of itself. The second is far worse for a moderation system, because there is no
confidence threshold at which you could have caught the mistake.

**What this notebook does.** Trains each model once at seed 42 at its best configuration,
saves the per-example predicted probabilities to `writing/probabilities/`, and computes
everything else from those files. The probabilities are saved so this never has to be
re-run: confusion matrices, the confidence score, calibration plots and any future
per-example analysis can all be rebuilt from the CSVs without touching a GPU.

Runtime: roughly 20 to 30 minutes on an RTX A5000 for all three transformers.

In [2]:
# Mount Google Drive on Colab only; locally this is a no-op.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not on Colab; using the local project folder.')

Not on Colab; using the local project folder.


In [3]:
from huggingface_hub import notebook_login; notebook_login()

In [4]:
# ---------------------------------------------------------------- environment
import os, random, gc, time, re
from pathlib import Path

# Keep Hugging Face downloads on the local drive, not the network home share.
HF_HOME = Path(os.environ.get('HINGLISH_HATE_HF_HOME',
                              Path(os.environ.get('LOCALAPPDATA', Path.home())) / 'huggingface'))
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'hub')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Optional Hugging Face authentication for gated models (IndicBERT v1).
# Set HF_TOKEN in the environment after accepting the model terms on the Hub.
# Never paste a real token into this notebook.
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print('Hugging Face token not set; gated models will be skipped.')

import numpy as np, pandas as pd, torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

# ROOT: env var or the local project folder first, then the Colab / Google Drive locations.
_candidates = [os.environ.get('HINGLISH_HATE_ROOT'),
               r'H:\Data\msc-hinglish-hate',
               Path.cwd().parent,
               '/content/drive/MyDrive/dissertation',
               '/content/dissertation',
               '/content']
ROOT = next((Path(c) for c in _candidates if c and (Path(c) / 'data').exists()), None)
if ROOT is None:
    raise FileNotFoundError(f'Could not find the project folder (with data/) in any of: {_candidates}. '
                            'Set HINGLISH_HATE_ROOT to point at it.')
print(f'ROOT: {ROOT.absolute()}')

import sys
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT))
from hinglish_hate import load_hasoc2022_threads, filter_romanised

DATA = ROOT / 'data'
W    = ROOT / 'writing'
PROB = W / 'probabilities'; PROB.mkdir(parents=True, exist_ok=True)
FIG  = W / 'figures';       FIG.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42

# FROZEN SPLIT - loaded, never re-created
train_df = pd.read_parquet(DATA/'splits'/'bohra_train.parquet')
test_df  = pd.read_parquet(DATA/'splits'/'bohra_test.parquet')
assert len(train_df) == 3659 and len(test_df) == 915, 'split changed'
cross_df = filter_romanised(load_hasoc2022_threads(DATA), include_mixed=True)

print(f'root {ROOT}')
print(f'device {DEVICE} | train {len(train_df)} | test {len(test_df)} | cross {len(cross_df)}')

Hugging Face token not set; gated models will be skipped.


c:\Users\k2558056\.conda\envs\msc-hinglish-hate\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: H:\Data\msc-hinglish-hate
root H:\Data\msc-hinglish-hate
device cuda | train 3659 | test 915 | cross 4306


## Hunter's metric

Two implementations, because they should agree and if they do not, one of them is wrong.

`confidence_score` is the literal transcription of his notes: compute `D`, attach a sign,
take the mean.

`confidence_score_identity` uses an algebraic simplification. For a binary classifier the
two probabilities sum to 1, so `P(Lower) = 1 - P(Higher)` and

$$D = P(\text{Higher}) - P(\text{Lower}) = 2P(\text{Higher}) - 1$$

When the prediction is correct, `P(Higher)` is the probability assigned to the true class,
so the signed contribution is $2P(\text{true}) - 1$. When the prediction is wrong, the true
class got `P(Lower)`, and the contribution is

$$-D = P(\text{Lower}) - P(\text{Higher}) = 2P(\text{Lower}) - 1 = 2P(\text{true}) - 1$$

The same expression either way. So the whole metric collapses to

$$\text{score} = \frac{2}{N}\sum_i P(\text{true class}_i) - 1$$

which is a linear rescaling of the mean probability assigned to the correct answer, mapped
from $[0, 1]$ onto $[-1, +1]$. That is worth saying in the Results chapter: it means the
score is a proper reading of average correct-class confidence, not an arbitrary construction,
and it is bounded, unlike the confidence *ratio* $P(\text{Higher})/P(\text{Lower})$ on the
second page of his notes, which diverges as $P(\text{Lower}) \to 0$. The ratio is computed
below as a median for completeness, but the difference score is the one to report.

In [5]:
# ---------------------------------------------------- Hunter's confidence metric
def confidence_difference(proba):
    """D = P(Higher) - P(Lower) for each row. Hunter's notes, page 2."""
    p = np.asarray(proba, dtype=float)
    return p.max(axis=1) - p.min(axis=1)


def confidence_score(y_true, proba):
    """Mean of (+D) when the prediction is correct and (-D) when it is wrong.

    Literal transcription of Hunter's notes, page 1.
    """
    p = np.asarray(proba, dtype=float)
    y = np.asarray(y_true, dtype=int)
    D = p.max(axis=1) - p.min(axis=1)
    sign = np.where(p.argmax(axis=1) == y, 1.0, -1.0)
    return float((sign * D).mean())


def confidence_score_identity(y_true, proba):
    """2 * (mean probability assigned to the true class) - 1.

    Algebraically identical to confidence_score for two classes. Kept as a check.
    """
    p = np.asarray(proba, dtype=float)
    y = np.asarray(y_true, dtype=int)
    return float(2.0 * p[np.arange(len(y)), y].mean() - 1.0)


# --- check against the worked examples in Hunter's own notes -------------------
assert abs(confidence_difference([[0.8, 0.2]])[0] - 0.6) < 1e-12   # eg (1)
assert abs(confidence_difference([[0.1, 0.9]])[0] - 0.8) < 1e-12   # eg (2)
assert abs(confidence_difference([[0.5, 0.5]])[0] - 0.0) < 1e-12   # eg (3)
assert abs(confidence_score([1], [[0.2, 0.8]]) - 0.6) < 1e-12      # correct  -> +0.6
assert abs(confidence_score([0], [[0.2, 0.8]]) + 0.6) < 1e-12      # incorrect -> -0.6

# --- check the two implementations agree on random input ----------------------
rng = np.random.default_rng(0)
_p = rng.random((5000, 1)); _p = np.hstack([_p, 1 - _p]); _y = rng.integers(0, 2, 5000)
assert abs(confidence_score(_y, _p) - confidence_score_identity(_y, _p)) < 1e-12
print('metric implementation verified against Hunter\'s worked examples')


def confidence_breakdown(y_true, proba, name='', split=''):
    """The score plus the diagnosis behind it.

    A score near zero has two very different causes, and Hunter named both: the model
    may be right and wrong in equal measure, or it may simply be unconfident about
    everything. Splitting the mean D by correctness tells the two apart.
    """
    p  = np.asarray(proba, dtype=float)
    y  = np.asarray(y_true, dtype=int)
    yp = p.argmax(axis=1)
    D  = p.max(axis=1) - p.min(axis=1)
    ok = (yp == y)
    acc = float(ok.mean())
    mD_ok  = float(D[ok].mean())  if ok.any()    else 0.0
    mD_bad = float(D[~ok].mean()) if (~ok).any() else 0.0
    ratio  = p.max(axis=1) / np.clip(p.min(axis=1), 1e-12, None)
    return {
        'model': name,
        'split': split,
        'n': int(len(y)),
        'accuracy': round(acc, 3),
        'macro_f1': round(f1_score(y, yp, average='macro'), 3),
        'confidence_score': round(confidence_score(y, p), 3),
        'mean_D_correct': round(mD_ok, 3),
        'mean_D_wrong': round(mD_bad, 3),
        'pct_errors_confident': round(float((D[~ok] > 0.5).mean() * 100), 1) if (~ok).any() else 0.0,
        'pct_all_unconfident': round(float((D < 0.2).mean() * 100), 1),
        'median_conf_ratio': round(float(np.median(ratio)), 2),
    }

metric implementation verified against Hunter's worked examples


## Collecting predicted probabilities

Everything downstream reads from `writing/probabilities/`. Each file is one model on one
test set, with the text, the gold label and both class probabilities, one row per example.

Four test sets per model:

- `within` — the frozen 915-post Bohra test set
- `cross` — HASOC 2022 romanised, never seen in training
- `rom` — the within set under the romanisation attack at full rate
- `char` — the within set under the character-noise attack at full rate

The two attacked sets cost one extra inference pass each and answer a question the
macro-F1 drop cannot: when an attack breaks a model, does it break it into uncertainty
or into confident error?

In [6]:
# ------------------------------------------------------------- shared machinery
from hinglish_hate.attack import romanisation_attack, character_attack, attack_corpus

def tag_of(model_name):
    return re.sub(r'[^a-z0-9]+', '_', model_name.lower()).strip('_')


def save_probs(tag, split, texts, y, proba):
    out = pd.DataFrame({
        'text':   list(texts),
        'label':  np.asarray(y, dtype=int),
        'p_not':  np.asarray(proba, dtype=float)[:, 0],
        'p_hate': np.asarray(proba, dtype=float)[:, 1],
    })
    path = PROB / f'{tag}__{split}.csv'
    out.to_csv(path, index=False)
    return path


# the attacked copies of the test set, built once and reused by every model
ATTACKED = {}
for split, fn in [('rom', romanisation_attack), ('char', character_attack)]:
    adv = attack_corpus(test_df, fn, rate=1.0, seed=42)
    ATTACKED[split] = adv
    print(f'{split}: realised token change rate {adv["token_change_rate"].mean():.3f}')

rom: realised token change rate 0.762
char: realised token change rate 0.710


In [7]:
# ------------------------------------------------------------- baseline (TF-IDF)
from hinglish_hate import load_lexicon
from hinglish_hate.baseline import build_full_pipeline

terms = load_lexicon(DATA / 'hinglish_slur_lexicon.csv')
pipe  = build_full_pipeline(terms)
pipe.fit(train_df['text'], train_df['label'])

tag = 'tfidf_logreg'
for split, frame in [('within', test_df), ('cross', cross_df),
                     ('rom', ATTACKED['rom']), ('char', ATTACKED['char'])]:
    proba = pipe.predict_proba(frame['text'])
    save_probs(tag, split, frame['text'], frame['label'], proba)
    print(f'  {split:7s} macro-F1 {f1_score(frame["label"], proba.argmax(1), average="macro"):.3f}')
print('baseline probabilities saved')

  within  macro-F1 0.642
  cross   macro-F1 0.435
  rom     macro-F1 0.594
  char    macro-F1 0.480
baseline probabilities saved


In [8]:
# --------------------------------------------------------------- transformers
counts  = np.bincount(train_df['label'].values)
weights = torch.tensor(len(train_df) / (2.0 * counts), dtype=torch.float).to(DEVICE)

class PostDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len=128):
        self.t = list(texts); self.y = list(labels); self.tok = tok; self.m = max_len
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = self.tok(str(self.t[i]), truncation=True, padding='max_length',
                     max_length=self.m, return_tensors='pt')
        return {'input_ids': e['input_ids'].squeeze(0),
                'attention_mask': e['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.y[i], dtype=torch.long)}

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

NUM_WORKERS = 0  # Windows notebooks must not spawn Dataset workers


def collect(model_name, epochs, seed=SEED, lr=2e-5, bs=16):
    """Train once at the model's best configuration and save probabilities for
    all four test sets. Same protocol as notebook 13, so the numbers are comparable."""
    set_all_seeds(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    tr  = DataLoader(PostDataset(train_df['text'], train_df['label'], tok),
                     batch_size=bs, shuffle=True, num_workers=NUM_WORKERS)

    set_all_seeds(seed)   # again, right before the randomly initialised head is built
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(tr) * epochs
    sch   = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    lossf = torch.nn.CrossEntropyLoss(weight=weights)

    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for b in tr:
            logits = model(input_ids=b['input_ids'].to(DEVICE),
                           attention_mask=b['attention_mask'].to(DEVICE)).logits
            loss = lossf(logits, b['labels'].to(DEVICE)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()

    def predict_proba(dl):
        model.eval(); P, T = [], []
        with torch.no_grad():
            for b in dl:
                lg = model(input_ids=b['input_ids'].to(DEVICE),
                           attention_mask=b['attention_mask'].to(DEVICE)).logits
                P.append(torch.softmax(lg, dim=1).cpu().numpy())
                T.extend(b['labels'].numpy())
        return np.array(T), np.vstack(P)

    tag = tag_of(model_name)
    for split, frame in [('within', test_df), ('cross', cross_df),
                         ('rom', ATTACKED['rom']), ('char', ATTACKED['char'])]:
        dl = DataLoader(PostDataset(frame['text'], frame['label'], tok),
                        batch_size=bs * 2, num_workers=NUM_WORKERS)
        y, proba = predict_proba(dl)
        save_probs(tag, split, frame['text'], y, proba)
        print(f'  {split:7s} macro-F1 {f1_score(y, proba.argmax(1), average="macro"):.3f}')

    print(f'  {round(time.time() - t0)}s')
    del model; gc.collect(); torch.cuda.empty_cache()


CONFIGS = [('xlm-roberta-base', 3),
           ('google/muril-base-cased', 6),
           ('ai4bharat/indic-bert', 6)]     # v1, matching the epoch sweep

for name, ep in CONFIGS:
    print(f'\n=== {name} | {ep} epochs | seed {SEED} ===')
    try:
        collect(name, ep)
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {str(e)[:160]}')


=== xlm-roberta-base | 3 epochs | seed 42 ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 698.93it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  within  macro-F1 0.653
  cross   macro-F1 0.470
  rom     macro-F1 0.638
  char    macro-F1 0.601
  117s

=== google/muril-base-cased | 6 epochs | seed 42 ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15647.16it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

  within  macro-F1 0.654
  cross   macro-F1 0.367
  rom     macro-F1 0.599
  char    macro-F1 0.572
  205s

=== ai4bharat/indic-bert | 6 epochs | seed 42 ===


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 10261.04it/s]
[transformers] AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
predictions.decoder.bias         | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
classifier.bias                  | MISSING    | 
classifier.weight                | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

  within  macro-F1 0.652
  cross   macro-F1 0.371
  rom     macro-F1 0.603
  char    macro-F1 0.577
  158s


## Confusion matrices

Counts, not percentages, with row-normalised shading so the diagonal is comparable across
models that see different numbers of examples. Class 0 is not-hate, class 1 is hate.

In [9]:
# ------------------------------------------------------------ confusion matrices
LABELS = ['not hate', 'hate']

def plot_cm(ax, y, yp, title):
    cm = confusion_matrix(y, yp, labels=[0, 1])
    row = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    ax.imshow(row, cmap='Blues', vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]}\n{row[i, j]*100:.1f}%', ha='center', va='center',
                    fontsize=10, color='white' if row[i, j] > 0.5 else 'black')
    ax.set_xticks([0, 1]); ax.set_xticklabels(LABELS, fontsize=8)
    ax.set_yticks([0, 1]); ax.set_yticklabels(LABELS, fontsize=8, rotation=90, va='center')
    ax.set_xlabel('predicted', fontsize=8); ax.set_ylabel('actual', fontsize=8)
    ax.set_title(title, fontsize=9)
    return cm


def load_probs(tag, split):
    path = PROB / f'{tag}__{split}.csv'
    if not path.exists():
        return None
    d = pd.read_csv(path)
    return d, d['label'].values, d[['p_not', 'p_hate']].values


MODELS = [('tfidf_logreg', 'TF-IDF + LogReg'),
          ('xlm_roberta_base', 'XLM-RoBERTa'),
          ('google_muril_base_cased', 'MuRIL'),
          ('ai4bharat_indic_bert', 'IndicBERT v1')]
AVAILABLE = [(t, n) for t, n in MODELS if (PROB / f'{t}__within.csv').exists()]
print('models with saved probabilities:', [n for _, n in AVAILABLE])

for split, label in [('within', 'within-dataset (Bohra test)'),
                     ('cross',  'cross-dataset (HASOC 2022)')]:
    fig, axes = plt.subplots(1, len(AVAILABLE), figsize=(3.1 * len(AVAILABLE), 3.3))
    axes = np.atleast_1d(axes)
    for ax, (tag, name) in zip(axes, AVAILABLE):
        d, y, p = load_probs(tag, split)
        plot_cm(ax, y, p.argmax(1), name)
    fig.suptitle(f'Confusion matrices, {label}', fontsize=11)
    fig.tight_layout()
    out = FIG / f'confusion_grid_{split}.png'
    fig.savefig(out, dpi=200, bbox_inches='tight'); plt.close(fig)
    print('saved', out.name)

# individual matrices as well, for use as separate figures in the dissertation
for split in ['within', 'cross', 'rom', 'char']:
    for tag, name in AVAILABLE:
        got = load_probs(tag, split)
        if got is None: continue
        d, y, p = got
        fig, ax = plt.subplots(figsize=(3.2, 3.2))
        plot_cm(ax, y, p.argmax(1), f'{name}, {split}')
        fig.tight_layout()
        fig.savefig(FIG / f'cm_{tag}__{split}.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('individual confusion matrices saved')

models with saved probabilities: ['TF-IDF + LogReg', 'XLM-RoBERTa', 'MuRIL', 'IndicBERT v1']
saved confusion_grid_within.png
saved confusion_grid_cross.png
individual confusion matrices saved


In [10]:
# ------------------------------------------------- the numbers behind each matrix
rows = []
for split in ['within', 'cross']:
    for tag, name in AVAILABLE:
        d, y, p = load_probs(tag, split)
        cm = confusion_matrix(y, p.argmax(1), labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        rows.append({'model': name, 'split': split,
                     'true_not': tn, 'false_hate': fp, 'missed_hate': fn, 'true_hate': tp,
                     'hate_recall': round(tp / max(tp + fn, 1), 3),
                     'hate_precision': round(tp / max(tp + fp, 1), 3),
                     'not_recall': round(tn / max(tn + fp, 1), 3)})
cm_table = pd.DataFrame(rows)
cm_table.to_csv(W / 'confusion_counts.csv', index=False)
display(cm_table)

,model,split,true_not,false_hate,missed_hate,true_hate,hate_recall,hate_precision,not_recall
0,TF-IDF + LogReg,within,415,168,140,192,0.578,0.533,0.712
1,XLM-RoBERTa,within,448,135,155,177,0.533,0.567,0.768
2,MuRIL,within,479,104,174,158,0.476,0.603,0.822
3,IndicBERT v1,within,446,137,154,178,0.536,0.565,0.765
4,TF-IDF + LogReg,cross,1817,282,1865,342,0.155,0.548,0.866
5,XLM-RoBERTa,cross,1632,467,1669,538,0.244,0.535,0.778
6,MuRIL,cross,2068,31,2118,89,0.040,0.742,0.985
7,IndicBERT v1,cross,2040,59,2102,105,0.048,0.640,0.972


## The confidence score

`confidence_score` is Hunter's number. The three columns after it are the diagnosis.

- **mean_D_correct** — how sure the model is when it is right
- **mean_D_wrong** — how sure it is when it is wrong
- **pct_errors_confident** — the share of mistakes made with `D > 0.5`, meaning the model
  put at least 75% probability on the wrong class

Hunter's hypothesis, in his words, was that "a lot of the wrong ones are very low
confidence". `mean_D_wrong` tests it directly. If it is well below `mean_D_correct`, the
model knows when it is struggling, and a confidence threshold would let a moderation system
route those cases to a human instead of acting on them. If the two are close, it does not.

The score decomposes exactly as

    confidence_score = accuracy * mean_D_correct - (1 - accuracy) * mean_D_wrong

which is printed below as a check.

In [11]:
# ----------------------------------------------------------- confidence results
rows = []
for split in ['within', 'cross', 'rom', 'char']:
    for tag, name in AVAILABLE:
        got = load_probs(tag, split)
        if got is None: continue
        d, y, p = got
        rows.append(confidence_breakdown(y, p, name=name, split=split))

conf = pd.DataFrame(rows)
conf.to_csv(W / 'confidence_scores.csv', index=False)

for split in ['within', 'cross', 'rom', 'char']:
    sub = conf[conf['split'] == split]
    if sub.empty: continue
    print(f'\n--- {split} ---')
    display(sub[['model', 'accuracy', 'macro_f1', 'confidence_score',
                 'mean_D_correct', 'mean_D_wrong', 'pct_errors_confident',
                 'pct_all_unconfident', 'median_conf_ratio']].to_string(index=False))

# decomposition check
print('\ndecomposition check (should match confidence_score to 3dp):')
for r in rows:
    lhs = r['accuracy'] * r['mean_D_correct'] - (1 - r['accuracy']) * r['mean_D_wrong']
    flag = 'ok' if abs(lhs - r['confidence_score']) < 0.005 else 'MISMATCH'
    print(f"  {r['model']:18s} {r['split']:7s} {lhs:+.3f} vs {r['confidence_score']:+.3f}  {flag}")


--- within ---


'          model  accuracy  macro_f1  confidence_score  mean_D_correct  mean_D_wrong  pct_errors_confident  pct_all_unconfident  median_conf_ratio\nTF-IDF + LogReg     0.663     0.642             0.149           0.353         0.254                  12.0                 34.9               1.86\n    XLM-RoBERTa     0.683     0.653             0.152           0.335         0.243                   7.6                 37.3               1.63\n          MuRIL     0.696     0.654             0.259           0.606         0.537                  71.9                  5.4               4.67\n   IndicBERT v1     0.682     0.652             0.172           0.421         0.361                  19.6                 17.6               2.27'


--- cross ---


'          model  accuracy  macro_f1  confidence_score  mean_D_correct  mean_D_wrong  pct_errors_confident  pct_all_unconfident  median_conf_ratio\nTF-IDF + LogReg     0.501     0.435             0.002           0.327         0.326                  17.0                 29.7               1.96\n    XLM-RoBERTa     0.504     0.470             0.003           0.338         0.338                  19.1                 22.7               1.99\n          MuRIL     0.501     0.367             0.002           0.625         0.623                  94.9                  1.2               4.66\n   IndicBERT v1     0.498     0.371            -0.001           0.346         0.346                   0.2                  9.3               2.25'


--- rom ---


'          model  accuracy  macro_f1  confidence_score  mean_D_correct  mean_D_wrong  pct_errors_confident  pct_all_unconfident  median_conf_ratio\nTF-IDF + LogReg     0.652     0.594             0.105           0.282         0.226                   6.3                 41.1               1.67\n    XLM-RoBERTa     0.666     0.638             0.101           0.261         0.220                   3.6                 44.6               1.57\n          MuRIL     0.658     0.599             0.212           0.606         0.546                  75.7                  4.7               4.64\n   IndicBERT v1     0.690     0.603             0.156           0.380         0.340                   6.7                 12.1               2.26'


--- char ---


'          model  accuracy  macro_f1  confidence_score  mean_D_correct  mean_D_wrong  pct_errors_confident  pct_all_unconfident  median_conf_ratio\nTF-IDF + LogReg     0.637     0.480             0.136           0.380         0.294                  14.5                 23.9               2.12\n    XLM-RoBERTa     0.643     0.601             0.091           0.264         0.220                   3.7                 43.8               1.60\n          MuRIL     0.621     0.572             0.156           0.589         0.553                  75.8                  4.8               4.59\n   IndicBERT v1     0.673     0.577             0.143           0.375         0.336                   7.4                 11.8               2.27'


decomposition check (should match confidence_score to 3dp):
  TF-IDF + LogReg    within  +0.148 vs +0.149  ok
  XLM-RoBERTa        within  +0.152 vs +0.152  ok
  MuRIL              within  +0.259 vs +0.259  ok
  IndicBERT v1       within  +0.172 vs +0.172  ok
  TF-IDF + LogReg    cross   +0.001 vs +0.002  ok
  XLM-RoBERTa        cross   +0.003 vs +0.003  ok
  MuRIL              cross   +0.002 vs +0.002  ok
  IndicBERT v1       cross   -0.001 vs -0.001  ok
  TF-IDF + LogReg    rom     +0.105 vs +0.105  ok
  XLM-RoBERTa        rom     +0.100 vs +0.101  ok
  MuRIL              rom     +0.212 vs +0.212  ok
  IndicBERT v1       rom     +0.157 vs +0.156  ok
  TF-IDF + LogReg    char    +0.135 vs +0.136  ok
  XLM-RoBERTa        char    +0.091 vs +0.091  ok
  MuRIL              char    +0.156 vs +0.156  ok
  IndicBERT v1       char    +0.143 vs +0.143  ok


In [12]:
# ------------------------------------------------- confidence distribution figure
fig, axes = plt.subplots(1, len(AVAILABLE), figsize=(3.4 * len(AVAILABLE), 3.2), sharey=True)
axes = np.atleast_1d(axes)
bins = np.linspace(0, 1, 21)
for ax, (tag, name) in zip(axes, AVAILABLE):
    d, y, p = load_probs(tag, 'within')
    D  = confidence_difference(p)
    ok = (p.argmax(1) == y)
    ax.hist(D[ok],  bins=bins, alpha=0.65, label='correct',   color='#2b7bba')
    ax.hist(D[~ok], bins=bins, alpha=0.65, label='incorrect', color='#d1495b')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('confidence difference D', fontsize=8)
    ax.tick_params(labelsize=8)
axes[0].set_ylabel('number of test posts', fontsize=8)
axes[0].legend(fontsize=8)
fig.suptitle('Confidence on correct and incorrect predictions, within-dataset test set', fontsize=11)
fig.tight_layout()
fig.savefig(FIG / 'confidence_distribution.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('saved confidence_distribution.png')

# score comparison across test conditions
piv = conf.pivot(index='model', columns='split', values='confidence_score')
piv = piv[[c for c in ['within', 'cross', 'rom', 'char'] if c in piv.columns]]
fig, ax = plt.subplots(figsize=(6.4, 3.4))
piv.plot(kind='bar', ax=ax, width=0.78, colormap='tab20c')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel("Hunter's confidence score"); ax.set_xlabel('')
ax.tick_params(axis='x', rotation=12, labelsize=8)
ax.legend(title='test set', fontsize=8, title_fontsize=8)
fig.tight_layout()
fig.savefig(FIG / 'confidence_score_by_condition.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('saved confidence_score_by_condition.png')

saved confidence_distribution.png
saved confidence_score_by_condition.png


## What to take to the next meeting

Three things come out of this notebook, and they answer the question Hunter has been
circling since August: why the results are not better.

**1. The confusion matrices show where the loss is.** Read `hate_recall` against
`not_recall` in `confusion_counts.csv`. If hate recall is much the lower of the two,
the models are not failing uniformly, they are failing on the minority class, which
is the systematic pattern Hunter suspected.

**2. The confidence score says what kind of failure it is.** A score near zero with
`mean_D_wrong` well below `mean_D_correct` is the good version of a mediocre result: the
model is unsure exactly where it is wrong. A score near zero with the two means close
together is the bad version. Say which one it is, in those terms, because that is the
distinction the metric was designed to make.

**3. The cross-dataset and attacked columns extend it.** If the confidence score falls
further than macro-F1 does under attack, the attack is not merely degrading accuracy, it
is making the model confidently wrong, which is a stronger claim for Objective 4 than the
F1 drop alone.

The remaining piece is the one this notebook cannot supply. The error analysis in notebook 12
found that 60% of the missed hate posts are political criticism or arguably mislabelled.
That is the ceiling: the models are being marked against a label definition that conflates
two different phenomena. The confusion matrices locate the loss, the confidence score
characterises it, and the error analysis explains it.

### Files produced

| Path | Contents |
|---|---|
| `writing/probabilities/*.csv` | per-example probabilities, four test sets per model |
| `writing/confusion_counts.csv` | the four cells of each matrix plus recall and precision |
| `writing/confidence_scores.csv` | the confidence score and its breakdown |
| `writing/figures/confusion_grid_within.png` | all models side by side, within-dataset |
| `writing/figures/confusion_grid_cross.png` | all models side by side, cross-dataset |
| `writing/figures/cm_*.png` | individual matrices for use as separate figures |
| `writing/figures/confidence_distribution.png` | D for correct against incorrect predictions |
| `writing/figures/confidence_score_by_condition.png` | the score across all four test sets |